# Waste Sorting Classification with TrashNet

This notebook builds a practical waste-category classifier on the TrashNet image dataset using PyTorch, timm, and Albumentations.

## Project Overview

The goal is to classify images of everyday waste into categories that support downstream sorting decisions: cardboard, glass, metal, paper, plastic, and trash.

## Learning Objectives

- Build a real image-classification pipeline on downloaded data.
- Fine-tune a compact timm backbone with Albumentations augmentations.
- Evaluate model behavior with confusion matrices, class-wise metrics, and mistake analysis.

## Problem Statement

Waste streams contain visually similar materials, contamination, and inconsistent image quality. A useful classifier must separate classes that often overlap in appearance, such as paper vs cardboard or plastic vs trash.

## Why This Project Matters

Better category predictions can reduce sorting friction, improve recycling yield, and surface where a lightweight vision model breaks down in real operations.

## Dataset Overview

This project uses TrashNet, a small educational waste-sorting dataset with six image folders.

## Dataset Source and License Notes

Dataset source link: https://github.com/garythung/trashnet

The notebook downloads the dataset from the public source repository during execution. Review the source repository for the latest license and reuse terms before shipping derived assets.

### Practical Dataset Noise

TrashNet is useful, but noisy in ways that matter for deployment:

- Some items are mixed-material objects, so the label may reflect the dominant material rather than every visible component.
- Recyclability varies by municipality, which means the image label is not the same as a universal disposal rule.
- Backgrounds, lighting, shadows, and camera angles vary enough to create shortcut signals.
- Paper, cardboard, plastic, and trash can overlap visually when objects are dirty, crumpled, torn, or partially occluded.
- A single object may look different when flattened, crushed, wet, reflective, or contaminated with food residue.

The error analysis later in the notebook focuses on these practical noise patterns rather than treating every mistake as pure model failure.

## Environment Setup

This cell ensures the notebook has the required packages. It installs missing packages directly in the active environment instead of assuming they already exist.

In [ ]:
import importlib
import subprocess
import sys

REQUIRED_PACKAGES = {
    'albumentations': 'albumentations',
    'nbformat': 'nbformat',
    'scikit-learn': 'sklearn',
    'seaborn': 'seaborn',
    'timm': 'timm',
}

for package_name, import_name in REQUIRED_PACKAGES.items():
    try:
        importlib.import_module(import_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package_name])

print('Environment setup complete.')

## Imports

These imports cover data access, visualization, augmentation, model training, and evaluation.

In [ ]:
import json
import math
import os
import random
import shutil
import urllib.request
import zipfile
from pathlib import Path

import albumentations as A
from albumentations.pytorch import ToTensorV2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image
import seaborn as sns
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
import timm

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('deep')

## Configuration / Constants

This section keeps runtime choices explicit and reproducible. The notebook uses a small timm backbone, real dataset download, and a short training schedule that still exercises the full pipeline on real data.

In [ ]:
SEED = 42
IMAGE_SIZE = 160
BATCH_SIZE = 32
NUM_EPOCHS = 1 if not torch.cuda.is_available() else 3
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
MODEL_NAME = 'resnet18'
DATASET_URL = 'https://github.com/garythung/trashnet/archive/refs/heads/master.zip'
PROJECT_DIR = Path.cwd()
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts'
DATA_DIR = PROJECT_DIR / 'data'
ARCHIVE_PATH = DATA_DIR / 'trashnet.zip'
EXTRACT_DIR = DATA_DIR / 'trashnet_extracted'
METRICS_PATH = PROJECT_DIR / 'metrics.json'

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
DATA_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print({'device': str(DEVICE), 'epochs': NUM_EPOCHS, 'project_dir': str(PROJECT_DIR)})

## Dataset Download and Loading

The dataset is downloaded inside the notebook and extracted locally. This notebook does not rely on preexisting copies and will fail loudly if the real dataset cannot be retrieved.

In [ ]:
if not ARCHIVE_PATH.exists():
    print(f'Downloading TrashNet from {DATASET_URL}')
    urllib.request.urlretrieve(DATASET_URL, ARCHIVE_PATH)

candidate_paths = [
    EXTRACT_DIR / 'trashnet-master' / 'data' / 'dataset-resized',
    EXTRACT_DIR / 'trashnet-master' / 'data' / 'dataset-resized.zip',
    EXTRACT_DIR / 'trashnet-main' / 'data' / 'dataset-resized',
    EXTRACT_DIR / 'trashnet-main' / 'data' / 'dataset-resized.zip',
    EXTRACT_DIR / 'trashnet' / 'data' / 'dataset-resized',
    EXTRACT_DIR / 'trashnet' / 'data' / 'dataset-resized.zip',
]

dataset_dir = None
dataset_zip_path = None
for candidate in candidate_paths:
    if candidate.exists() and candidate.is_dir():
        dataset_dir = candidate
        break
    if candidate.exists() and candidate.suffix == '.zip':
        dataset_zip_path = candidate

if dataset_dir is None and dataset_zip_path is None:
    if EXTRACT_DIR.exists():
        shutil.rmtree(EXTRACT_DIR)
    EXTRACT_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ARCHIVE_PATH, 'r') as zip_ref:
        zip_ref.extractall(EXTRACT_DIR)
    for candidate in candidate_paths:
        if candidate.exists() and candidate.is_dir():
            dataset_dir = candidate
            break
        if candidate.exists() and candidate.suffix == '.zip':
            dataset_zip_path = candidate

if dataset_dir is None and dataset_zip_path is not None:
    extracted_dataset_dir = dataset_zip_path.with_suffix('')
    if not extracted_dataset_dir.exists():
        with zipfile.ZipFile(dataset_zip_path, 'r') as zip_ref:
            zip_ref.extractall(dataset_zip_path.parent)
    if extracted_dataset_dir.exists():
        dataset_dir = extracted_dataset_dir

if dataset_dir is None:
    raise RuntimeError('TrashNet extracted successfully, but dataset-resized was not found in the downloaded archive.')

CLASS_NAMES = sorted([path.name for path in dataset_dir.iterdir() if path.is_dir()])
print({'dataset_dir': str(dataset_dir), 'classes': CLASS_NAMES})

## Data Validation Checks

Before training, validate the real downloaded files. These checks surface common operational issues such as empty classes, unsupported extensions, duplicate paths, or corrupted images.

In [ ]:
records = []
valid_suffixes = {'.jpg', '.jpeg', '.png'}

for class_name in CLASS_NAMES:
    class_dir = dataset_dir / class_name
    for image_path in sorted(class_dir.iterdir()):
        if image_path.suffix.lower() in valid_suffixes:
            records.append({'path': image_path, 'label_name': class_name})

data = pd.DataFrame(records)
if data.empty:
    raise RuntimeError('No images were found after downloading TrashNet.')

data['label'] = data['label_name'].map({name: index for index, name in enumerate(CLASS_NAMES)})
data['relative_path'] = data['path'].map(lambda path: path.relative_to(dataset_dir).as_posix())

duplicates = int(data['relative_path'].duplicated().sum())
missing_files = int((~data['path'].map(Path.exists)).sum())
corrupt_files = []

for image_path in data['path']:
    try:
        with Image.open(image_path) as image:
            image.verify()
    except Exception:
        corrupt_files.append(str(image_path))

class_counts = data['label_name'].value_counts().sort_index()
empty_classes = [class_name for class_name in CLASS_NAMES if class_counts.get(class_name, 0) == 0]

if missing_files != 0 or duplicates != 0 or corrupt_files or empty_classes:
    raise RuntimeError({
        'missing_files': missing_files,
        'duplicates': duplicates,
        'corrupt_files': corrupt_files[:5],
        'empty_classes': empty_classes,
    })

print({
    'num_images': len(data),
    'num_classes': len(CLASS_NAMES),
    'class_counts': class_counts.to_dict(),
})

## Exploratory Data Analysis

EDA helps reveal class imbalance and visual variation.

## Train/Validation/Test Split Strategy

TrashNet does not ship with a fixed split, so this notebook creates a stratified 70/15/15 split to preserve class balance while keeping validation and test estimates separate.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
class_counts.plot(kind='bar', ax=axes[0], color='teal')
axes[0].set_title('Class Distribution')
axes[0].set_xlabel('Class')
axes[0].set_ylabel('Image Count')

sample_rows = data.groupby('label_name', group_keys=False).head(2).reset_index(drop=True)
grid_cols = 4
grid_rows = math.ceil(len(sample_rows) / grid_cols)
axes[1].axis('off')
fig.tight_layout()
plt.show()

figure, sample_axes = plt.subplots(grid_rows, grid_cols, figsize=(14, 3 * grid_rows))
sample_axes = np.array(sample_axes).reshape(grid_rows, grid_cols)
for axis in sample_axes.ravel():
    axis.axis('off')

for index, row in sample_rows.iterrows():
    image = Image.open(row['path']).convert('RGB')
    axis = sample_axes.ravel()[index]
    axis.imshow(image)
    axis.set_title(row['label_name'])
    axis.axis('off')

figure.suptitle('Sample TrashNet Images', fontsize=16)
figure.tight_layout()
plt.show()

train_frame, temp_frame = train_test_split(
    data,
    test_size=0.30,
    random_state=SEED,
    stratify=data['label'],
)
val_frame, test_frame = train_test_split(
    temp_frame,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_frame['label'],
)

for split_name, frame in [('train', train_frame), ('val', val_frame), ('test', test_frame)]:
    print(split_name, len(frame), frame['label_name'].value_counts().sort_index().to_dict())

## Preprocessing and Augmentation Strategy

The training pipeline uses Albumentations for moderate geometric and color perturbations. The goal is to improve robustness to the real-world messiness of waste images without fabricating unrealistic views.

## Baseline Approach

A majority-class baseline shows what happens if the system always predicts the most frequent training class. The learned model should beat that baseline by a meaningful margin.

## Main Model / Workflow

The main model is a timm `resnet18` classifier fine-tuned on TrashNet. It is intentionally compact so the notebook remains practical on CPU while still demonstrating transfer learning.

In [ ]:
train_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.HorizontalFlip(p=0.5),
    A.Affine(scale=(0.95, 1.05), translate_percent=(-0.05, 0.05), rotate=(-12, 12), p=0.5),
    A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.02, p=0.5),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(IMAGE_SIZE, IMAGE_SIZE),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
])

class TrashNetDataset(Dataset):
    def __init__(self, frame, transform):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, index):
        row = self.frame.iloc[index]
        image = Image.open(row['path']).convert('RGB')
        image = np.array(image)
        tensor = self.transform(image=image)['image']
        return tensor, int(row['label'])

train_dataset = TrashNetDataset(train_frame, train_transform)
val_dataset = TrashNetDataset(val_frame, eval_transform)
test_dataset = TrashNetDataset(test_frame, eval_transform)

pin_memory = DEVICE.type == 'cuda'
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=pin_memory)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_memory)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=pin_memory)

majority_class = train_frame['label'].mode().iloc[0]
baseline_predictions = np.full(shape=len(test_frame), fill_value=int(majority_class))
baseline_accuracy = accuracy_score(test_frame['label'], baseline_predictions)
baseline_macro_f1 = f1_score(test_frame['label'], baseline_predictions, average='macro')

model = timm.create_model(MODEL_NAME, pretrained=True, num_classes=len(CLASS_NAMES))
model = model.to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
print({'baseline_accuracy': baseline_accuracy, 'baseline_macro_f1': baseline_macro_f1})

## Training Loop or Fine-Tuning Pipeline

The fine-tuning loop tracks loss and accuracy on both train and validation splits. Short runs are intentional here: the notebook is designed to be executable on ordinary hardware while still training on the real dataset.

In [ ]:
def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    total_examples = 0
    all_targets = []
    all_predictions = []

    for inputs, targets in loader:
        inputs = inputs.to(DEVICE)
        targets = targets.to(DEVICE)

        with torch.set_grad_enabled(training):
            logits = model(inputs)
            loss = criterion(logits, targets)
            if training:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        predictions = logits.argmax(dim=1)
        batch_size = targets.size(0)
        total_loss += loss.item() * batch_size
        total_examples += batch_size
        all_targets.extend(targets.detach().cpu().numpy().tolist())
        all_predictions.extend(predictions.detach().cpu().numpy().tolist())

    average_loss = total_loss / total_examples
    average_accuracy = accuracy_score(all_targets, all_predictions)
    return average_loss, average_accuracy

history = []
best_state = None
best_val_accuracy = -1.0

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_accuracy = run_epoch(train_loader, training=True)
    with torch.no_grad():
        val_loss, val_accuracy = run_epoch(val_loader, training=False)

    history.append({
        'epoch': epoch,
        'train_loss': train_loss,
        'train_accuracy': train_accuracy,
        'val_loss': val_loss,
        'val_accuracy': val_accuracy,
    })

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    print(history[-1])

if best_state is None:
    raise RuntimeError('Training did not produce a best checkpoint.')

model.load_state_dict(best_state)
history_frame = pd.DataFrame(history)
history_frame

## Inference Examples

Inference examples show how the model behaves on held-out images rather than training samples.

## Evaluation

The evaluation focuses on accuracy, balanced accuracy, macro F1, the confusion matrix, and per-class precision/recall/F1. Macro F1 matters here because minority classes are easy to ignore in small datasets.

In [ ]:
@torch.no_grad()
def predict_dataset(loader):
    model.eval()
    all_targets = []
    all_predictions = []
    all_probabilities = []

    for inputs, targets in loader:
        inputs = inputs.to(DEVICE)
        logits = model(inputs)
        probabilities = torch.softmax(logits, dim=1)
        predictions = probabilities.argmax(dim=1)

        all_targets.extend(targets.numpy().tolist())
        all_predictions.extend(predictions.cpu().numpy().tolist())
        all_probabilities.extend(probabilities.cpu().numpy().tolist())

    return np.array(all_targets), np.array(all_predictions), np.array(all_probabilities)

y_true, y_pred, y_prob = predict_dataset(test_loader)
test_accuracy = accuracy_score(y_true, y_pred)
test_balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
test_macro_f1 = f1_score(y_true, y_pred, average='macro')
report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
conf_matrix = confusion_matrix(y_true, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].plot(history_frame['epoch'], history_frame['train_accuracy'], marker='o', label='Train Accuracy')
axes[0].plot(history_frame['epoch'], history_frame['val_accuracy'], marker='o', label='Val Accuracy')
axes[0].set_title('Training History')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()

sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title('Test Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

fig.tight_layout()
plt.show()

display(pd.DataFrame(report).transpose().round(3))

inference_rows = test_frame.reset_index(drop=True).head(8).copy()
figure, axes = plt.subplots(2, 4, figsize=(16, 8))
for axis in axes.ravel():
    axis.axis('off')

for index, row in inference_rows.iterrows():
    image = Image.open(row['path']).convert('RGB')
    tensor = eval_transform(image=np.array(image))['image'].unsqueeze(0).to(DEVICE)
    probabilities = torch.softmax(model(tensor), dim=1).cpu().numpy()[0]
    prediction_index = int(np.argmax(probabilities))
    confidence = float(probabilities[prediction_index])
    axis = axes.ravel()[index]
    axis.imshow(image)
    axis.set_title(f
)
    axis.axis('off')

figure.suptitle('Inference Examples on Held-Out Test Images', fontsize=16)
figure.tight_layout()
plt.show()

metrics = {
    'dataset': 'TrashNet',
    'dataset_source': 'https://github.com/garythung/trashnet',
    'model_name': MODEL_NAME,
    'device': str(DEVICE),
    'num_epochs': NUM_EPOCHS,
    'image_size': IMAGE_SIZE,
    'class_names': CLASS_NAMES,
    'train_size': int(len(train_frame)),
    'val_size': int(len(val_frame)),
    'test_size': int(len(test_frame)),
    'baseline_accuracy': float(baseline_accuracy),
    'baseline_macro_f1': float(baseline_macro_f1),
    'test_accuracy': float(test_accuracy),
    'test_balanced_accuracy': float(test_balanced_accuracy),
    'test_macro_f1': float(test_macro_f1),
    'classification_report': report,
    'dataset_noise_note': 'TrashNet contains label ambiguity, contamination, mixed materials, and visually overlapping categories that affect recyclability judgments.',
}

with open(METRICS_PATH, 'w', encoding='utf-8') as file:
    json.dump(metrics, file, indent=2)

print({
    'baseline_accuracy': round(baseline_accuracy, 4),
    'test_accuracy': round(test_accuracy, 4),
    'test_macro_f1': round(test_macro_f1, 4),
    'metrics_path': str(METRICS_PATH),
})

## Error Analysis

This step inspects held-out mistakes to separate systematic weaknesses from noisy or ambiguous labels. In TrashNet, common failure modes include dirty paper looking like trash, cardboard pieces resembling paper, and reflective metal or glass confusing the model under harsh lighting.

In [ ]:
error_frame = test_frame.reset_index(drop=True).copy()
error_frame['true_name'] = [CLASS_NAMES[index] for index in y_true]
error_frame['pred_name'] = [CLASS_NAMES[index] for index in y_pred]
error_frame['is_error'] = error_frame['true_name'] != error_frame['pred_name']
mistakes = error_frame[error_frame['is_error']].head(8).reset_index(drop=True)

if mistakes.empty:
    print('No misclassifications were found in the current test predictions.')
else:
    figure, axes = plt.subplots(2, 4, figsize=(16, 8))
    for axis in axes.ravel():
        axis.axis('off')

    for index, row in mistakes.iterrows():
        image = Image.open(row['path']).convert('RGB')
        axis = axes.ravel()[index]
        axis.imshow(image)
        axis.set_title(f
)
        axis.axis('off')

    figure.suptitle('Misclassified Test Images', fontsize=16)
    figure.tight_layout()
    plt.show()

mistakes[['relative_path', 'true_name', 'pred_name']].head(10)

## Limitations

- TrashNet is small, so variance across splits can be meaningful.
- The label represents a category, not a complete municipal recycling policy.
- Some images are visually ambiguous because the object is dirty, damaged, or mixed-material.

## How to Improve This Project

- Add more current smartphone photos from the target deployment environment.
- Use a larger pretrained timm backbone and longer schedule when GPU time is available.
- Add calibration and a reject option for low-confidence predictions.
- Track municipality-specific recyclable vs non-recyclable rules separately from material category.

## Production Considerations

- Surface uncertainty rather than forcing a confident label.
- Expect domain shift from classroom-style dataset images to messy bin-line photos.
- Pair image classification with metadata or OCR if packaging labels are available.

## Common Mistakes

- Treating TrashNet labels as universal disposal advice.
- Ignoring class imbalance and relying only on raw accuracy.
- Hiding ambiguous examples instead of documenting them as practical dataset noise.

## Mini Challenge / Exercises

1. Replace `resnet18` with a stronger timm backbone and compare macro F1.
2. Add label smoothing or focal loss and check whether minority-class recall improves.
3. Build a confidence-threshold rule that routes uncertain items to manual review.

## Final Summary / Key Takeaways

This notebook trains a real waste-sorting classifier on the real TrashNet download, exports metrics, and shows why practical dataset noise matters. The main lesson is that visually plausible labels can still be operationally ambiguous, so error analysis and uncertainty handling matter as much as top-line accuracy.